# humor-me decisions: the model track

Evidence for the judge, SFT and DPO. Experiment numbers are global across the `decisions*.ipynb` notebooks; the index is [decisions.ipynb](decisions.ipynb).

**How to use it**
- Results load from `reports/metrics/*.json` (cached by parameters), so nothing recomputes unless you ask.
- Every experiment is a `run(**params)` called through `sim.cache.run_or_load`; its module's docstring says how to re-record it.
- ✅/❌ lines check the claim each decision depends on: all ✅ means every decision here still holds.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"))
from decisions_common import RERUN, check, show

## Exp 17 DPO judge gate (2026-11-26)

**Decision (D-116):** does Tingyu's DPO-trained rJokes judge (Qwen2.5-7B-Instruct + LoRA `dpo_judge_v1`) pick the more-upvoted of two jokes above chance and above the same model untrained? If it does, it stands ready as D-102's fallback judge. D-102 is unchanged: the D-95 judge labels DPO's pairs if it passes its own gate.
**Where it fits:** the model track's pairwise judge ([Bradley–Terry](../docs/wiki/Bradley-Terry.md), [judge/](../judge/README.md)). Pairs: `explorer/build_judge_pairs.py`; training: `explorer/train_dpo.py --stage judge`; scoring: `explorer/eval_judge.py` on Explorer.
**Method:** both models answer "Which joke is funnier? A or B" for the 5,318 rJokes **test** pairs, each in both orders (10,636 prompts). A pair scores 1 if right in both orders, 0.5 if right in one, 0 if wrong in both; CIs are over pairs. The cluster's result files hold pair hashes and scores, no joke text; `python -m sim.judge_gate --record` recomputes every number from them and stores numbers only.
**Bar, fixed before this run:** the trained judge's 95% CI lower bound above 0.5 **and** the paired gain over the untrained model with its CI lower bound above 0. Half-widths are at most about ±0.013 (pair sd ≤ 0.5).
**What is made up:** nothing; it is a measurement on held-out data. Its limit is the label: r/Jokes upvotes on written jokes, not a live room or the user's taste.
**How to read:** "same answer in both orders" is consistency, not correctness; "right when consistent" says how often a consistent answer is right.

In [2]:
from sim import judge_gate as jg

res = jg.stored()
if res is None:
    show("**Not recorded yet.** Copy `runs/judge_eval/{base,adapter}.json` from Explorer to `data/results/exp17_judge_eval/`, run `python -m sim.judge_gate --record`, then re-run this notebook.")
else:
    show(jg.table(res))
    u, t, d = res["arms"]["untrained"], res["arms"]["trained"], res["paired_diff"]
    check(f"The trained judge is above chance: accuracy {t['accuracy']:.3f}, CI lower bound {t['ci95'][0]:.3f} > 0.5", t["ci95"][0] > 0.5)
    check(f"It beats the untrained model on the same pairs: gain {d['mean']:+.3f}, CI lower bound {d['ci95'][0]:+.3f} > 0", d["ci95"][0] > 0)
    check("The recorded gate is what the fixed rule gives", res["gate_pass"] == jg.gate(t, d) == True)
    check("Every pair of the test split was scored in both orders, none unscored, for both models",
          u["n_pairs"] == t["n_pairs"] == d["n_pairs"] == res["n_test_pairs"] and u["n_unscored"] == t["n_unscored"] == 0
          and all(a["right_both"] + a["right_one"] + a["right_neither"] == a["n_pairs"] for a in (u, t)))
    check("The eval script's own summary matches the numbers recomputed from its per-pair scores", all(res["eval_script_agrees"].values()))
    check(f"Training made the judge more order-consistent ({u['consistent_share']:.0%} → {t['consistent_share']:.0%}); consistency is not correctness: right on {t['accuracy_when_consistent']:.0%} of its consistent pairs",
          t["consistent_share"] > u["consistent_share"] and t["accuracy_when_consistent"] < 1)

| | Untrained Qwen2.5-7B | DPO judge (`dpo_judge_v1`) |
|---|---|---|
| Accuracy, per pair (95% CI) | 0.527 (0.518 to 0.537) | 0.695 (0.684 to 0.706) |
| Right in both orders | 28% (1,476) | 60% (3,213) |
| Right in one order only (flipped) | 50% (2,655) | 18% (967) |
| Wrong in both orders | 22% (1,187) | 21% (1,138) |
| Same answer both orders | 50.1% | 81.8% |
| Right when consistent | 55.4% | 73.8% |
| Share answered "A" (position bias) | 27.3% | 43.0% |

Paired gain, trained minus untrained, same 5,318 pairs: **+0.168** (95% CI +0.157 to +0.179); trained better on 2,161 pairs, same on 2,489, worse on 668. Gate: **pass**.

✅ **The trained judge is above chance: accuracy 0.695, CI lower bound 0.684 > 0.5**

✅ **It beats the untrained model on the same pairs: gain +0.168, CI lower bound +0.157 > 0**

✅ **The recorded gate is what the fixed rule gives**

✅ **Every pair of the test split was scored in both orders, none unscored, for both models**

✅ **The eval script's own summary matches the numbers recomputed from its per-pair scores**

✅ **Training made the judge more order-consistent (50% → 82%); consistency is not correctness: right on 74% of its consistent pairs**